# ParkSense AI — Exploratory Data Analysis & Data Cleaning
Owner: Eric (backend-ml-analysis)

Cleans the raw 298k violation dataset and exports a processed Parquet file for model training.

In [ ]:
import os
import pandas as pd
import numpy as np
import h3

# Paths
RAW_CSV_PATH = "../data/jan to may police violation_anonymized791b166.csv"
PROCESSED_DIR = "../data/processed"
OUTPUT_PARQUET_PATH = os.path.join(PROCESSED_DIR, "violations_clean.parquet")

print("Starting data cleaning and preprocessing...")

# 1. Load dataset
print(f"Loading raw CSV from {RAW_CSV_PATH}...")
df = pd.read_csv(RAW_CSV_PATH)
print(f"Loaded {len(df):,} rows.")

# 2. Clean coordinates
print("Cleaning coordinates...")
df = df.dropna(subset=["latitude", "longitude"])
df = df[(df["latitude"] > 12.0) & (df["latitude"] < 14.0)]
df = df[(df["longitude"] > 77.0) & (df["longitude"] < 79.0)]

# 3. Clean datetimes
print("Processing timestamps...")
df["timestamp"] = pd.to_datetime(df["created_datetime"], format="mixed")
df.drop(columns=["created_datetime"], inplace=True)

# 4. Generate H3 Resolution 8 cells
print("Computing H3 spatial bins...")
df["h3_cell"] = [h3.latlng_to_cell(lat, lng, 8) for lat, lng in zip(df["latitude"], df["longitude"])]

# 5. Map baseline vehicle blockage factors
print("Mapping vehicle blockage factors...")
blockage_map = {
    "CAR": 1.5,
    "MAXI-CAB": 1.6,
    "SCOOTER": 0.5,
    "MOTOR CYCLE": 0.5,
    "THREE WHEELER": 1.0,
    "TEMPO": 1.8,
    "TRUCK": 2.2,
    "BUS": 2.5,
    "TRACTOR": 2.0
}
df["vehicle_type"] = df["vehicle_type"].fillna("UNKNOWN").str.upper()
df["blockage_factor"] = df["vehicle_type"].map(blockage_map).fillna(1.0)

# 6. Temporal weight
df["hour"] = df["timestamp"].dt.hour
df["temporal_weight"] = np.where(df["hour"].isin([8, 9, 10, 17, 18, 19, 20]), 1.3, 1.0)

# 7. Baseline CIS Score (scaled 0-100)
print("Computing baseline CIS scores...")
df["cis_score"] = (df["blockage_factor"] * df["temporal_weight"].astype(float) * 30.0).round(2)
df["cis_score"] = df["cis_score"].clip(upper=100.0)

# 8. Clean validation status
df["validation_status"] = df["validation_status"].fillna("pending").str.lower()

# 9. Format vehicle number
df["vehicle_number"] = df["vehicle_number"].fillna("UNKNOWN").str.upper().str.replace("-", "")

# Rename id to violation_id
if "id" in df.columns:
    df.rename(columns={"id": "violation_id"}, inplace=True)

# Keep only required columns
required_cols = [
    "violation_id", "vehicle_number", "vehicle_type", "violation_type",
    "latitude", "longitude", "timestamp", "police_station", "junction_name",
    "validation_status", "cis_score", "h3_cell"
]
df = df[[c for c in required_cols if c in df.columns]]

# Save
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Saving cleaned dataset to {OUTPUT_PARQUET_PATH}...")
df.to_parquet(OUTPUT_PARQUET_PATH, index=False)
print(f"Successfully saved {len(df):,} rows.")